# Упражнение 3, част 2: линейна регресия

**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Цел на упражнението](#cel)
2. [Файлове в упражнението](#failove)
3. [Подготовка и стартиране](#podgotovka)
4. [Линейна регресия](#linear-regression)
5. [Част 1. Предиктори и отговор](#chast-1)
6. [Част 2. Проста линейна регресия](#chast-2)
7. [Част 3. Остатъци и точност на модела](#chast-3)
8. [Част 4. Множествена линейна регресия](#chast-4)
9. [Част 5. Качествени предиктори](#chast-5)
10. [Част 6. Взаимодействие](#chast-6)
11. [Обобщение](#obobshtenie)
12. [Задача](#zadacha)

<a id="cel"></a>
## Цел на упражнението

В първото упражнение напаснахме права за цена на жилище според площта.
Във второто отделихме тренировъчни и тестови редове и подготвихме признаци.
Сега ще разгледаме по-подробно **линейната регресия** от лекцията (Тема 3):
как се намира правата, колко точни са коефициентите и прогнозите ѝ, как се
тълкуват коефициентите при един и при няколко предиктора и как моделът се
разширява с качествени предиктори и взаимодействия.

Работим със същите данни като в лекцията, затова числата в тетрадката
могат да се сравнят със слайдовете. Основният пример са рекламните данни,
а упражнението следва въпросите от лекцията:

| Въпрос от лекцията | Къде в упражнението |
|---|---|
| Може ли да се предскажат продажбите чрез бюджетите за телевизия, радио и вестници? | части 1 и 2 |
| Колко точни са оценките и прогнозите? | част 3 |
| Кои медии допринасят за продажбите? | част 4 |
| Има ли взаимодействие между рекламните канали? | част 6 |

Част 5 използва данните Credit от лекцията за качествените предиктори, а
задачата използва данните Auto за нелинейните зависимости. Всеки модел
сравняваме и с постоянна прогноза, която служи за отправна точка.

<a id="failove"></a>
## Файлове в упражнението

Пътищата са относителни спрямо папката на тази тетрадка.

| Файл | За какво служи |
|---|---|
| `Упражнение_3_част_2.ipynb` | обяснения, примери и задачи |
| `ЗА_ВКЪЩИ_част_2.ipynb` | задачи за самостоятелна работа |
| `solution_2.py` | решения на задачите ДА ОПИТАМЕ и на финалната задача |
| `data/advertising.csv` | 200 пазара, три рекламни бюджета и продажби |
| `data/credit.csv` | 400 клиенти, баланс по кредитната карта и десет предиктора |
| `data/auto.csv` | 397 модела коли, разход на гориво, мощност и други характеристики (за задачата) |
| `../environment.yml` | обща среда при работа с `conda` |
| `../requirements.txt` | общ списък с пакети при работа с `pip` |

Трите файла са Advertising.csv, Credit.csv и Auto.csv от [сайта на
авторите на *An Introduction to Statistical Learning*](https://www.statlearning.com/resources-python),
същите данни като в лекцията. Първата колона на `advertising.csv` е номер
на ред, а не рекламен бюджет. Разходите са в хиляди долари, а продажбите
са в хиляди бройки.

<a id="podgotovka"></a>
## Подготовка и стартиране

Използваме средата, създадена по време на първото упражнение.
От папката на **третото** упражнение обновяваме пакетите по
избрания начин, ако е необходимо. Първо активираме средата:

| Среда | Команди |
|---|---|
| Anaconda Prompt | 1. `conda activate ml2627`<br/>2. `conda env update -f ../environment.yml` |
| PowerShell и `pip` | 1. `..\.venv\Scripts\Activate.ps1`<br/>2. `python -m pip install -r ../requirements.txt` |

След това, от същата папка, стартираме `jupyter lab`, отваряме
`Упражнение_3_част_2.ipynb` и избираме ядрото **Python (МС 26/27)**.
Ако пакет се намира в терминала, но липсва в тетрадката, проверяваме
избраното ядро. Ако CSV файлът не се намира, проверяваме от коя папка
сме стартирали Jupyter.

In [ ]:
import solution_2

%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("работна папка:", os.getcwd())

<a id="linear-regression"></a>
## Линейната регресия (Linear Regression)

**Линейната регресия** е метод за **обучение с учител (supervised learning)**. При този вид обучение разполагаме с примери, за които знаем както входните данни - **признаците**, така и правилния резултат - **целевата променлива (target)**. Задачата на модела е да научи зависимостта между тях, така че след това да може да предсказва целта за нови, невиждани до този момент примери.

Линейната регресия се използва, когато **целта е числова стойност**. Например можем да предсказваме цена на жилище, температура, време за доставка или количество продажби.

В нашия пример:

- **признакът** е бюджетът за TV реклама;
- **целта** са продажбите;
- всеки ред от данните описва известен рекламен бюджет и реално измерените при него продажби.

Например тренировъчните данни могат да съдържат наблюдения като:

| TV бюджет (хил. долари) | Продажби (хил. бройки) |
|---:|---:|
| 20 | 7 |
| 50 | 9 |
| 100 | 12 |
| 150 | 16 |

От тези примери моделът трябва да научи как обикновено се променят продажбите, когато се променя рекламният бюджет.

Методът се нарича **линейна** регресия, защото прогнозата е сбор от признаците, умножени по коефициенти, плюс свободен член. При един признак това е права линия. Ако например научената зависимост е

**прогнозирани продажби = 5 + 0.07 × TV бюджет**,

то при TV бюджет 100 моделът би прогнозирал:

**5 + 0.07 × 100 = 12 хиляди продажби.**

Ако увеличим бюджета от 100 на 110, прогнозата ще се увеличи с 0.7. Ако го увеличим още веднъж с 10 - от 110 на 120 - прогнозата отново ще се увеличи с 0.7. *Именно това е идеята за **линейна зависимост**: еднаква промяна на признака води до еднаква промяна на прогнозата.*

Колоните на модела могат да бъдат и изчислени от признаците, например квадратът на признака в задачата накрая. Тогава графиката на прогнозата вече не е права, но моделът остава сбор от колони, умножени по коефициенти, и се обучава по същия начин. Затова и полиномиалната регресия е линейна регресия.

Разбира се, в реалните данни точките почти никога не лежат точно върху една права. Например при TV бюджет 100, реалните продажби може да са 11, а моделът да прогнозира 12. При друг пример може да прогнозира 15, а реалната стойност да е 16. Това е нормално - върху продажбите влияят и много други фактори.

Затова при обучението линейната регресия търси не права, която минава през всяка точка, а **права, чиито прогнози са възможно най-близо до известните стойности от тренировъчните данни**.

След обучението можем да дадем на модела TV бюджет, който не е виждал преди, например 130, и да поискаме прогноза за продажбите. Това е основната цел на машинното обучение: **да научи закономерност от известни примери и да я използва за нови примери**.

За да проверим дали моделът действително е научил полезна зависимост, обикновено разделяме данните на:

- **тренировъчно множество** - от него моделът се учи;
- **тестово множество** - върху него проверяваме колко добре предсказва примери, които не е използвал при обучението.

Важно е да разграничаваме **регресия** от **класификация**. И двете са обучение с учител, но при регресията целта е число, например продажби = 12.4, докато при класификацията целта е категория, например "ирис setosa" или "ирис versicolor". Линейният модел за класификация, логистичната регресия, е в лекцията за Тема 4.

### Кога линейната регресия е полезна

Линейната регресия изглежда проста, но е
изключително полезна. Тя е добър избор, когато:

- **целта е число и зависи от признаците приблизително линейно**, или
  формата може да се опише с няколко изчислени колони, като квадрат или
  произведение;
- **трябва да се разбере зависимостта, а не само да се прогнозира**: всеки
  коефициент казва с колко се променя прогнозата, когато признакът нарасне
  с 1, а останалите не се променят, например колко продажби носят още 1000
  долара за TV;
- **трябва да се знае колко може да се вярва на резултата**: стандартните
  грешки и доверителните интервали показват кои коефициенти са надеждни;
- **данните са малко**: моделът има малко коефициенти и се обучава
  надеждно дори от 150 реда, колкото са тренировъчните пазари тук;
- **трябва отправна точка за сравнение**: обучава се за миг, няма параметри
  за настройка и винаги дава едно и също решение. По-сложен модел си
  струва само ако е по-точен от нея.

Къде линейният модел не стига, показват
[точка 12 "Остатъците спрямо прогнозите"](#t12),
[точка 20 "Връзка, която линейният модел не вижда"](#t20) и
[задачата](#zadacha).

<a id="chast-1"></a>
## Част 1. Предиктори и отговор

Данните в **advertising.csv** описват 200 пазара: рекламните бюджети и продажбите на всеки от тях. Признаците са бюджетите за реклама в **TV**, **radio** и **newspaper**, а целевата променлива е **sales** - реализираните продажби. В лекцията признаците се наричат **предиктори** (X₁, X₂, X₃), а целевата променлива **отговор** (Y). Линейната регресия предполага линейна зависимост между тях.

Задачата е задача за **обучение с учител и регресия**: използваме известните рекламни бюджети и продажби, за да обучим модел, който да предсказва продажбите при нови стойности на рекламния бюджет. В най-простия вариант можем да използваме само бюджета за **TV реклама** като единствен признак. Това е нашата първа стъпка. Но нека най-напред се запознаем с данните.

### 1. Зареждане на таблицата

`pd.read_csv` връща pandas `DataFrame`. С `index_col=0` използваме
първата колона от файла за индекс и не я включваме сред признаците.
`display` показва таблицата удобно в тетрадката. Импортирахме го от
`IPython.display`.

In [ ]:
df = pd.read_csv("data/advertising.csv", index_col=0)
print("Форма:", df.shape)
print("Пропуски по колони:")
print(df.isna().sum())
display(df.head())
display(df.describe().round(1))

### 2. Кои са признаците и целта

`TV`, `radio` и `newspaper` са рекламни бюджети (признаците). `sales` е
непрекъсната числова цел. Това е задача за **регресия**.
Един ред е един пример. Например `TV=100` означава
100 хиляди долара телевизионен бюджет.

Първо ще разгледаме поотделно връзката между **продажбите** и всеки от трите признака -
бюджетите за **TV**, **radio** и **newspaper** реклама. За всеки признак ще начертаем
облак от точки, за да видим дали с увеличаването на рекламния бюджет се наблюдава тенденция
продажбите също да се променят. Така можем визуално да добием представа кои признаци изглеждат
свързани с продажбите.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, feature in zip(axes, ["TV", "radio", "newspaper"]):
    ax.scatter(df[feature], df["sales"], alpha=0.65, s=24)
    ax.set(xlabel=f"{feature} (хил. долари)", ylabel="продажби (хил. бр.)")
fig.suptitle("Продажби и рекламни бюджети")
fig.tight_layout()
plt.show()

> **Въпрос.** Коя от трите колони според графиките очакваме
> да дава добра прогноза сама?

<details><summary>Отговор</summary>

Очакваме `TV` да е най-полезна сама, защото точките около възходящата
тенденция са най-събрани. При `radio` също има връзка, но точките са
по-разпилени, а при `newspaper` връзката е най-слаба. Същото казва и
лекцията. Това е предположение от графиката. Ще го проверим с модела и
с отделените тестови редове.

</details>

<a id="chast-2"></a>
## Част 2. Проста линейна регресия

Под **проста** се разбира с един предиктор. Тук признакът е 
бюджетът за TV, а целта са продажбите. Измерените
продажби не е нужно да лежат точно върху една права. Обучението
избира права, която описва възможно най-добре тренировъчните данни.
После проверяваме прогнозите върху отделени тестови редове.

### 3. Отделяне на редове за проверка

Ще построим модел, който описва връзката между бюджета за **TV реклама** и **продажбите** с права линия. В лекцията простата линейна регресия е **Y ≈ β₀ + β₁X + ε**: β₀ е свободният член, β₁ е наклонът, а ε е грешката, т.е. всичко, което правата не описва. След обучението получаваме

**ŷ = β̂₀ + β̂₁ · TV**

Шапката означава оценена стойност. Тук **ŷ** е продажбата, която моделът прогнозира за даден TV бюджет. Числата **β̂₀** и **β̂₁** определят къде се намира правата и колко е наклонена:

- **β̂₀** е прогнозата на модела при `TV = 0`;
- **β̂₁** показва с колко се променя прогнозата за продажбите, когато TV бюджетът се увеличи с една единица.

Например, ако моделът е научил `β̂₀ = 7` и `β̂₁ = 0.05`, при `TV = 100` прогнозата ще бъде `7 + 0.05 · 100 = 12`. Ако увеличим TV бюджета с още една единица, прогнозата ще се увеличи с `0.05`.

Но стойностите на **β̂₀** и **β̂₁** не ги задаваме ние. По време на **обучението** моделът ги избира така, че правата да описва възможно най-добре примерите от тренировъчното множество, тоест прогнозите ѝ да бъдат възможно най-близо до реалните продажби.

При това моделът вижда **само тренировъчните редове**. Тестовите данни не участват в избора на правата. Запазваме ги, за да проверим след обучението колко добре моделът предсказва продажбите за данни, които не е виждал преди.

`X` е таблица с признаци, `y` е целта. Използваме едно и също разделяне за всички модели по-долу, за да са сравними резултатите им.

In [ ]:
from sklearn.model_selection import train_test_split

features = ["TV", "radio", "newspaper"]
X = df[features]
y = df["sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print("Тренировъчни редове:", X_train.shape)
print("Тестови редове:", X_test.shape)

### 4. Постоянна прогноза за сравнение

`DummyRegressor(strategy="mean")` научава само средните тренировъчни
продажби и ги прогнозира за всеки нов ред. Това е същата идея за
постоянна прогноза, която вече използвахме. Ако обучената права
не се справя по-добре, телевизионният бюджет не ни е помогнал
в тази проверка.

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean").fit(X_train[["TV"]], y_train)
baseline_test_pred = baseline.predict(X_test[["TV"]])
print("Постоянна прогноза:", round(baseline.constant_[0, 0], 2),
      "хил. продажби")

### 5. Обучение на правата

`LinearRegression().fit(...)` намира свободния член и наклона,
които минимизират сумата от квадратите на **тренировъчните** остатъци.
Това е методът на **най-малките квадрати** от лекцията
([точка 8 "Остатъци и RSS"](#t8)).
`predict` прилага научената права към нови стойности на `TV`.
За една колона подаваме таблица с форма `(n, 1)`, затова пишем
`X_train[["TV"]]` с двойни квадратни скоби.

In [ ]:
from sklearn.linear_model import LinearRegression

tv_model = LinearRegression().fit(X_train[["TV"]], y_train)
tv_test_pred = tv_model.predict(X_test[["TV"]])

print("Свободен член β̂₀:", round(tv_model.intercept_, 3))
print("Наклон β̂₁ за TV:", round(tv_model.coef_[0], 4))
print("Прогноза за TV=100:",
      round(tv_model.predict(pd.DataFrame({"TV": [100.0]}))[0], 2))

Наклон около `0.047` означава: при още 1000 долара за телевизия
прогнозата се увеличава с около `0.047` хиляди продажби, тоест с около
47 бройки. Това е **връзка в наблюдаваните данни**, а не доказан
причинен ефект.
Свободният член е пресечната точка на правата с оста на продажбите, т.е. `TV=0`.

> **Въпрос.** Какво ще отпечата този нов код? Използваме единиците
> от таблицата: бюджети в хиляди долари и продажби в хиляди бройки.
>
> ```python
> intercept = 4.0
> slope = 0.08
> print(intercept + slope * 25, intercept + slope * 75)
> ```

<details><summary>Отговор</summary>

Получаваме `6.0 10.0`. Увеличение на `TV` с 50 хиляди долара
променя прогнозата с `0.08 × 50 = 4` хиляди бройки в този пример.

</details>

<a id="t6"></a>
### 6. Правата от лекцията

В лекцията правата е обучена с всички 200 пазара: β̂₀ = 7.03 и
β̂₁ = 0.0475, т.е. още 1000 долара за TV реклама се свързват средно с около
47.5 допълнителни продажби. Тук моделът видя само 150 тренировъчни пазара,
затова числата са близки, но не съвпадат. Всяка извадка от пазари дава
малко по-различна права. Колко се различават правите, показва
стандартната грешка в
[точка 10 "Стандартна грешка и доверителен интервал"](#t10).

`lecture_model` е обучен с всички редове, както в лекцията. Използваме го
в част 3 за точността на оценките, но не и за проверката върху тестовите
редове, защото ги е виждал.

In [ ]:
lecture_model = LinearRegression().fit(df[["TV"]], df["sales"])
print("Всички 200 пазара: β̂₀ =", round(lecture_model.intercept_, 4),
      "| β̂₁ =", round(lecture_model.coef_[0], 4))
print("150 тренировъчни:  β̂₀ =", round(tv_model.intercept_, 4),
      "| β̂₁ =", round(tv_model.coef_[0], 4))

### 7. Правата върху графиката

Точките са наблюденията; линията е прогнозата на модела. Сивите
вертикални отсечки от тренировъчните точки до правата са остатъците. Тестовите
точки са показани **след** обучението, само за да видим как правата се държи
върху отделените пазари.

In [ ]:
tv_grid = pd.DataFrame({
    "TV": np.linspace(X_train["TV"].min(), X_train["TV"].max(), 100)
})
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.vlines(X_train["TV"], y_train, tv_model.predict(X_train[["TV"]]),
          color="gray", linewidth=0.8, alpha=0.6)
ax.scatter(X_train["TV"], y_train, alpha=0.55, label="тренировъчни")
ax.scatter(X_test["TV"], y_test, alpha=0.8, label="тестови")
ax.plot(tv_grid["TV"], tv_model.predict(tv_grid), color="black",
        linewidth=2, label="обучена права")
ax.set(xlabel="TV бюджет (хил. долари)",
       ylabel="продажби (хил. бр.)")
ax.legend()
plt.show()

<a id="chast-3"></a>
## Част 3. Остатъци и точност на модела

<a id="t8"></a>
### 8. Остатъци и RSS

**Остатъкът** е действителната стойност минус прогнозата:
`eᵢ = yᵢ - ŷᵢ`. Положителен остатък означава, че моделът е
прогнозирал по-малко от наблюдаваното, т.е. точка над линията,
отрицателен означава точка под нея. Остатъкът е вертикалното разстояние
от линията до точката, а знакът показва посоката: 
нагоре е плюс, надолу е минус. **Сумата от квадратите на остатъците**
(residual sum of squares, **RSS**) е сборът
`Σ eᵢ²` върху тренировъчните редове и измерва общото несъответствие
между наблюденията и правата. Методът на най-малките квадрати
избира β̂₀ и β̂₁ така, че този сбор да е минимален: всяка друга права
има по-голям RSS.

Следващата клетка сравнява RSS на научената права и на правата от
лекцията върху същите 150 тренировъчни пазара.

In [ ]:
tv_train_pred = tv_model.predict(X_train[["TV"]])
train_residuals = y_train.to_numpy() - tv_train_pred
train_rss = np.sum(train_residuals ** 2)
lecture_train_rss = np.sum(
    (y_train.to_numpy() - lecture_model.predict(X_train[["TV"]])) ** 2
)

print("Първите пет остатъка:", train_residuals[:5].round(2))
print("RSS на научената права:", round(train_rss, 2))
print("RSS на правата от лекцията:", round(lecture_train_rss, 2))

Правата от лекцията е най-добрата за всички 200 пазара, но за тези 150
научената права има малко по-малък RSS. Разликата е малка, защото двете
прави почти съвпадат.

> **Въпрос.** Какви са остатъците и RSS за този нов пример?
>
> ```python
> actual = np.array([8.0, 11.0, 15.0])
> predicted = np.array([10.0, 10.0, 14.0])
> residuals = actual - predicted
> print(residuals, np.sum(residuals ** 2))
> ```

<details><summary>Отговор</summary>

Остатъците са `[-2., 1., 1.]`, а RSS е `4 + 1 + 1 = 6`.
Отрицателният първи остатък означава, че прогнозата е по-висока
от действителната стойност.

</details>

In [ ]:
# --- ДА ОПИТАМЕ 1 ---
#
# Може ли друга права да има по-малък RSS от научената? Прекарайте права през
# тренировъчните пазари с най-малък и с най-голям TV бюджет и изчислете RSS
# върху тренировъчните редове. После запазете свободния член β̂₀ и сменяйте
# само наклона, например от 0.035 до 0.06. При кой наклон RSS е най-малък?

solution_2.to_test_1(X_train, y_train, tv_model)

#### Коефициентите по формула

ДА ОПИТАМЕ 1 търси най-малкия RSS с проби: пресмята RSS за много наклони
и избира най-малкия. `LinearRegression` не прави проби. При един
предиктор дъното на RSS се намира с математика, наведнъж.

RSS е функция на двете неизвестни β₀ и β₁:

`RSS(β₀, β₁) = Σ(yᵢ - β₀ - β₁xᵢ)²`

В най-ниската точка на функция на две променливи двете **частни
производни** са нула. Частната производна по β₀ е обикновената
производна, при която β₁ се смята за константа, и обратно:

`∂RSS/∂β₀ = -2 · Σ(yᵢ - β₀ - β₁xᵢ) = 0`

`∂RSS/∂β₁ = -2 · Σxᵢ(yᵢ - β₀ - β₁xᵢ) = 0`

**Първото уравнение.** Скобата е остатъкът eᵢ, т.е. сумата на остатъците
е нула. След деление на n остава ȳ - β₀ - β₁x̄ = 0, откъдето

`β̂₀ = ȳ - β̂₁ · x̄`

Тук x̄ и ȳ са средният бюджет и средните продажби. Правата минава през
точката (x̄, ȳ): при среден бюджет прогнозата е средните продажби.

**Второто уравнение.** След заместване на β₀ скобата става
(yᵢ - ȳ) - β₁(xᵢ - x̄). Сборът на тези скоби е нула, защото Σ(yᵢ - ȳ) = 0
и Σ(xᵢ - x̄) = 0. Затова множителят xᵢ може да се замени с xᵢ - x̄, без
сборът да се промени. Остава
Σ(xᵢ - x̄)(yᵢ - ȳ) - β₁ · Σ(xᵢ - x̄)² = 0, откъдето

`β̂₁ = Σ(xᵢ - x̄)(yᵢ - ȳ) / Σ(xᵢ - x̄)²`

- **Числителят** събира произведенията на отклоненията от средните. Пазар
  с бюджет над средния и продажби над средните дава положително
  произведение. Ако такива пазари преобладават, наклонът е положителен.
- **Знаменателят** показва колко са разпилени бюджетите около средния.

Следващата клетка пресмята двете формули с NumPy върху същите 150
тренировъчни пазара.

In [ ]:
tv = X_train["TV"].to_numpy()
sales = y_train.to_numpy()
tv_mean, sales_mean = tv.mean(), sales.mean()

slope_formula = (np.sum((tv - tv_mean) * (sales - sales_mean))
                 / np.sum((tv - tv_mean) ** 2))
intercept_formula = sales_mean - slope_formula * tv_mean
residuals_formula = sales - (intercept_formula + slope_formula * tv)

print("Средни: x̄ =", round(tv_mean, 2), "| ȳ =", round(sales_mean, 2))
print("По формулата:     β̂₀ =", round(intercept_formula, 4),
      "| β̂₁ =", round(slope_formula, 4))
print("LinearRegression: β̂₀ =", round(tv_model.intercept_, 4),
      "| β̂₁ =", round(tv_model.coef_[0], 4))
print("Сума на остатъците:", round(np.sum(residuals_formula), 10))
print("RSS:", round(np.sum(residuals_formula ** 2), 2))

Коефициентите съвпадат с тези на `LinearRegression`, а RSS е същият
като на научената права по-горе. Сумата на остатъците е нула, както
казва първото уравнение. Формулата дава точното дъно наведнъж, а пробите
в ДА ОПИТАМЕ 1 го намират само с точност до стъпката между наклоните.

При няколко предиктора идеята е същата: производните по всички
коефициенти се приравняват на нула. Получава се система от линейни
уравнения, по едно за всеки коефициент, и `LinearRegression` решава
нея. Затова резултатът винаги е един и същ и не зависи от начални
стойности.

> **Въпрос.** За следните три точки (1, 1), (2, 3) и (3, 2)
> коя права има най-малък RSS? Какви са β̂₀ и β̂₁ по двете формули?

<details><summary>Отговор</summary>

Средните са x̄ = 2 и ȳ = 2. Отклоненията от средните са -1, 0, 1 за x 
и -1, 1, 0 за y. Сборът от произведенията им е 1 + 0 + 0 = 1, а сборът
от квадратите на отклоненията на x е 1 + 0 + 1 = 2. Тогава 
β̂₁ = 1 / 2 = 0.5 и β̂₀ = 2 - 0.5 · 2 = 1, т.е. правата е 1 + 0.5x.

Прогнозите на правата са 1.5, 2 и 2.5, а остатъците са
1 - 1.5 = -0.5, 3 - 2 = 1 и 2 - 2.5 = -0.5. Остатъците не са същите
като отклоненията на y, защото се мерят от правата, а не от средното.
Сборът им е 0, а RSS = 0.25 + 1 + 0.25 = 1.5. Никоя друга права няма
по-малък RSS за тези точки. Правата не минава през нито една от тях,
но минава през (x̄, ȳ) = (2, 2).

</details>

<a id="t9"></a>
### 9. Точност на модела: RSE и R²

Лекцията описва точността на правата с две числа, изчислени с всички 200
пазара:

- **RSE** (остатъчната стандартна грешка, residual standard error)
  оценява стандартното отклонение на ε, т.е. с колко типично се
  отклоняват продажбите от правата: `RSE = √(RSS / (n - 2))`. Единицата е
  като на продажбите.
- **R²** е делът от изменчивостта на продажбите, обяснен от модела:
  `R² = (TSS - RSS) / TSS`, където `TSS = Σ(yᵢ - ȳ)²` е  общата сума от
  квадратите на отклоненията на y от средното (Total Sum of Squares).
  TSS е сумата от квадратите на остатъците при постоянна прогноза ȳ, а
  RSS е същата сума при правата. R² казва с каква част от TSS е по-малък
  RSS.
  При проста линейна регресия `R² = r²`, където r е корелацията между X и Y.

В лекцията RSE = 3.26 и R² = 0.612. Методът `score` на модела също връща R².

In [ ]:
n = len(df)
rss = np.sum((df["sales"] - lecture_model.predict(df[["TV"]])) ** 2)
tss = np.sum((df["sales"] - df["sales"].mean()) ** 2)
rse = np.sqrt(rss / (n - 2))
r = df["TV"].corr(df["sales"])

print("RSS:", round(rss, 1), "| TSS:", round(tss, 1))
print("RSE:", round(rse, 2))
print("R² = (TSS - RSS) / TSS:", round((tss - rss) / tss, 3))
print("r²:", round(r ** 2, 3))
print("score:", round(lecture_model.score(df[["TV"]], df["sales"]), 3))

TV рекламата обяснява 61.2% от изменчивостта на продажбите, а типичната
грешка е около 3.26 хиляди продажби. Останалите 38.8% зависят от неща,
които правата не вижда, например бюджета за радио.

> **Въпрос.** Колко са RSE и R² за трите точки (1, 1), (2, 3) и (3, 2)
> и правата 1 + 0.5x от въпроса в
> [точка 8 "Остатъци и RSS"](#t8)?

<details><summary>Отговор</summary>

Остатъците са -0.5, 1 и -0.5, а RSS = 1.5. Отклоненията на y от
средното ȳ = 2 са -1, 1 и 0, затова TSS = 1 + 1 + 0 = 2.

- RSE = √(RSS / (n - 2)) = √(1.5 / (3 - 2)) = √1.5 ≈ 1.22. Точките се
  отклоняват от правата типично с около 1.2 в единиците на y.
- R² = (TSS - RSS) / TSS = (2 - 1.5) / 2 = 0.25. Правата намалява сумата
  от квадратите на остатъците с 25% спрямо постоянната прогноза ȳ = 2:
  от TSS = 2 на RSS = 1.5.
- Проверка с корелацията:
  r = Σ(xᵢ - x̄)(yᵢ - ȳ) / √(Σ(xᵢ - x̄)² · Σ(yᵢ - ȳ)²) = 1 / √(2 · 2) = 0.5,
  т.е. r² = 0.25, колкото е R². Трите суми вече са пресметнати: 1 и 2 в
  точка 8, а третата е TSS = 2.

Делението е на n - 2, защото двата коефициента са нагласени по самите
точки. При две точки правата минава точно през тях, RSS = 0 и n - 2 = 0,
т.е. няма от какво да се оцени разсейването. Тук от трите точки остава
само една в повече, затова RSE е груба оценка.

</details>

<a id="t10"></a>
### 10. Стандартна грешка и доверителен интервал

С други 200 пазара правата щеше да е малко по-различна, както в
[точка 6 "Правата от лекцията"](#t6).
**Стандартната грешка** SE(β̂₁) показва колко се мени оценката при
повторни извадки. Тя зависи от σ² = Var(ε) и от разпределението на X:

`SE(β̂₁)² = σ² / Σ(xᵢ - x̄)²`

σ не е известна и се заменя с RSE. Приблизителният 95% доверителен
интервал е `β̂₁ ± 2 · SE(β̂₁)`. scikit-learn не изчислява стандартни
грешки и p-стойности (в ISLP за тях се използва библиотеката
statsmodels), затова SE се смята по формулата от лекцията.

Ако нулата е в интервала, данните допускат, че наклонът е 0, т.е. че
y изобщо не зависи от x, затова не може да се твърди, че има връзка,
но това не доказва, че връзка няма.

In [ ]:
slope = lecture_model.coef_[0]
se_slope = np.sqrt(rse ** 2 / np.sum((df["TV"] - df["TV"].mean()) ** 2))
print("β̂₁ =", round(slope, 4), "| SE(β̂₁) =", round(se_slope, 4))
print("95% интервал:", round(slope - 2 * se_slope, 4), "до",
      round(slope + 2 * se_slope, 4))

При още 1000 долара за TV продажбите растат средно с между 42 и 53
бройки. Нулата е далеч извън интервала: данните показват връзка между TV
рекламата и продажбите. В лекцията това е казано с p-стойност под 0.0001.

> **Въпрос.** Колко са SE(β̂₁) и приблизителният 95% доверителен
> интервал за наклона при трите точки (1, 1), (2, 3) и (3, 2) от
> [точка 9 "Точност на модела: RSE и R²"](#t9)? Може ли от тях да се
> твърди, че y расте с x?

<details><summary>Отговор</summary>

От предишните въпроси β̂₁ = 0.5, Σ(xᵢ - x̄)² = 2 и RSE = √1.5 ≈ 1.22.
σ се заменя с RSE:

- SE(β̂₁)² = RSE² / Σ(xᵢ - x̄)² = 1.5 / 2 = 0.75, т.е.
  SE(β̂₁) = √0.75 ≈ 0.87;
- интервалът е 0.5 ± 2 · 0.87 = 0.5 ± 1.73, т.е. от -1.23 до 2.23.

Интервалът съдържа нулата и дори отрицателни наклони. Три точки не
стигат, за да се твърди, че y расте с x: при друга тройка точки от
същия свят наклонът лесно може да излезе отрицателен. SE намалява,
когато точките са повече и стойностите на x са по-разпилени, защото
тогава Σ(xᵢ - x̄)² е по-голяма. Затова при TV с 200 пазара интервалът е
тесен и далеч от нулата.

</details>

In [ ]:
# --- ДА ОПИТАМЕ 2 ---
#
# Стандартната грешка показва колко се мени наклонът при повторни извадки.
# Други пазари нямаме, но можем да ги симулираме: изберете 200 пазара с
# връщане, df.sample(200, replace=True, random_state=i), и обучете правата.
# Повторете 1000 пъти, запазете наклоните и покажете хистограма. Колко е
# стандартното отклонение на наклоните? Близо ли е до SE(β̂₁) от формулата?

solution_2.to_test_2(df, se_slope)

### 11. Проверка върху тестовите редове

RSS, RSE и R² по-горе измерват напасването към данните, с които е обучен
моделът. За нови пазари използваме отделените тестови редове:

| Мярка | Как я четем |
|---|---|
| MAE | средна абсолютна грешка, в хиляди продадени бройки |
| RMSE | корен от средната квадратична грешка (MSE); наказва по-силно големите грешки |
| R² | `(TSS - RSS) / TSS` както в лекцията, но върху тестовите редове |

По-малки MAE и RMSE са по-добри. По-голямо R² е по-добро; върху тестови
данни то може да е и отрицателно, ако моделът греши повече от средната
тестова стойност. Не бъркаме тестовото R² с твърдение, че рекламата
причинява точно този дял от продажбите.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metric_row(name, actual, predicted):
    return pd.Series({
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }, name=name)

comparison = pd.DataFrame([
    metric_row("Постоянна прогноза", y_test, baseline_test_pred),
    metric_row("Само TV", y_test, tv_test_pred),
])
comparison.index.name = "модел"
display(comparison.round(3))

<a id="t12"></a>
### 12. Остатъците спрямо прогнозите

Изчертаваме тестовите остатъци спрямо прогнозите, както графиките на
остатъците в края на лекцията. При добър модел очакваме точки от двете
страни на нулата без ясен образец и с еднаква разсейка. Систематична форма
подсказва, че простата права не описва всичко важно в данните.

In [ ]:
test_residuals = y_test.to_numpy() - tv_test_pred
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(tv_test_pred, test_residuals, alpha=0.8)
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="прогноза (хил. бр.)",
       ylabel="остатък: действително - прогноза (хил. бр.)",
       title="Тестови остатъци на модела само с TV")
plt.show()

При малките прогнози остатъците са близо до нулата, а при големите
разсейката е много по-голяма: точките образуват фуния. Моделът само с TV
не вижда нещо, което има значение при големите TV бюджети.

In [ ]:
# --- ДА ОПИТАМЕ 3 ---
#
# Покажете шестте тестови пазара с най-голяма абсолютна грешка на модела само
# с TV: бюджетите им, действителните продажби, прогнозата и остатъка.
# Какво е общото между пазарите, за които моделът прогнозира твърде малко?
# А между тези, за които прогнозира твърде много?

solution_2.to_test_3(X_test, y_test, tv_test_pred)

<a id="chast-4"></a>
## Част 4. Множествена линейна регресия

### 13. Обучение с три предиктора

При p предиктора моделът от лекцията е `Y = β₀ + β₁X₁ + ... + βₚXₚ + ε`.
За рекламните данни:

`sales = β₀ + β₁ · TV + β₂ · radio + β₃ · newspaper + ε`

Всеки коефициент показва промяната на прогнозата при една единица
повече от съответния бюджет, **ако другите бюджети са фиксирани**.
Това е различно от наклона на права, обучена само с един признак.
Коефициентите отново минимизират RSS. При два предиктора моделът е
равнина, а остатъците са вертикалните разстояния от точките до нея.

Пазим същите тренировъчни и тестови пазари, за да сравним резултатите
с модела само с `TV`.

In [ ]:
multi_model = LinearRegression().fit(X_train, y_train)
multi_test_pred = multi_model.predict(X_test)

coefficients = pd.Series(multi_model.coef_, index=features)
print("Свободен член:", round(multi_model.intercept_, 3))
print("Коефициенти:")
print(coefficients.round(4))

comparison.loc["TV + radio + newspaper"] = metric_row(
    "TV + radio + newspaper", y_test, multi_test_pred
)
display(comparison.round(3))

Коефициентът на `newspaper` е близо до нула: при фиксирани `TV` и
`radio` прогнозата почти не се променя с бюджета за вестници. В лекцията
p-стойността му е 0.86, т.е. в този модел няма доказателство за връзка.

### 14. Прости и множествени коефициенти

В лекцията резултатите са с всички 200 пазара: свободен член 2.939 и
коефициенти 0.046 за TV, 0.189 за radio и -0.001 за newspaper. Следващата
клетка обучава този модел и го сравнява с трите прости регресии, всяка
само с един предиктор. Показва и корелациите между колоните, както
таблицата в лекцията.

In [ ]:
full_multi = LinearRegression().fit(df[features], df["sales"])
print("Свободен член:", round(full_multi.intercept_, 3))
simple_vs_multiple = pd.DataFrame({
    "проста регресия": [
        LinearRegression().fit(df[[feature]], df["sales"]).coef_[0]
        for feature in features
    ],
    "множествена регресия": full_multi.coef_,
}, index=features)
display(simple_vs_multiple.round(4))
display(df.corr().round(4))

В простата регресия `newspaper` има наклон 0.055: на пазарите с повече
реклама във вестници продажбите са по-високи. В множествения модел
коефициентът ѝ е почти нула. Корелацията между `radio` и `newspaper` е
0.35: където се харчи повече за радио, се харчи повече и за вестници.
Простата регресия приписва на вестниците част от ефекта на радиото. Когато
`radio` е в модела, вестниците не добавят почти нищо.

### 15. Прогнози и действителни стойности

Графиката сравнява тестовите прогнози на модела с трите бюджета с
действителните продажби. Диагоналът показва идеална прогноза. Всяка
точка над или под него е пазар, за който има грешка. Графиката показва размера
и посоката на грешките, които една обща метрика скрива.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(y_test, multi_test_pred, alpha=0.75)
lower = min(y_test.min(), multi_test_pred.min())
upper = max(y_test.max(), multi_test_pred.max())
ax.plot([lower, upper], [lower, upper], color="black", linewidth=1)
ax.set(xlabel="действителни продажби (хил. бр.)",
       ylabel="прогнозирани продажби (хил. бр.)")
plt.show()

In [ ]:
# --- ДА ОПИТАМЕ 4 ---
#
# При p предиктора има 2ᵖ възможни модела. Последователният избор напред от
# лекцията започва без предиктори и на всяка стъпка добавя този, който намалява
# RSS най-много. Направете го с трите бюджета върху всички 200 пазара.
# RSS винаги намалява, затова за спиране използвайте коригираното R²:
# 1 - (RSS / (n - p - 1)) / (TSS / (n - 1)), където p е броят на предикторите.
# Спрете, когато то намалее. Кои предиктори остават?

solution_2.to_test_4(df, features, X_train, X_test, y_train, y_test)

<a id="chast-5"></a>
## Част 5. Качествени предиктори

Някои предиктори са категории, а не числа. В лекцията примерът е наборът
Credit: балансът по кредитната карта (`Balance`) на 400 клиенти, шест
числови предиктора (доход, лимит, рейтинг, брой карти, възраст,
образование) и четири качествени: `Own` (собствено жилище), `Student`,
`Married` и `Region` (`East`, `South` или `West`). Категориите се
представят чрез **индикаторни променливи** с 0 и 1, както one-hot
кодирането от упражнение 2.

### 16. Двоичен качествен предиктор

За `Own` задаваме xᵢ = 1 за клиент със собствено жилище и xᵢ = 0 за клиент
без. Моделът е `balanceᵢ = β₀ + β₁xᵢ + εᵢ`. Тогава β₀ е средният баланс на
базовата група (без собствено жилище), а β₁ е разликата между двете групи.

In [ ]:
credit = pd.read_csv("data/credit.csv")
display(credit.head())

own = (credit["Own"] == "Yes").astype(int).to_frame("Own_Yes")
own_model = LinearRegression().fit(own, credit["Balance"])
print("β̂₀ =", round(own_model.intercept_, 2), "| β̂₁ =", round(own_model.coef_[0], 2))
print("Среден баланс по групи:")
print(credit.groupby("Own")["Balance"].mean().round(2))

β̂₀ е средният баланс без собствено жилище, а β̂₀ + β̂₁ е средният баланс
със собствено жилище. Разликата 19.73 е малка спрямо разсейката на
баланса. В лекцията p-стойността ѝ е 0.669: няма статистическо
доказателство за разлика между двете групи.

### 17. Качествен предиктор с повече нива

`Region` има три нива. При q категории са необходими q - 1 индикаторни
променливи: тук две, за `South` и за `West`. Третото ниво, `East`, е
**базовата категория**: за него и двете колони са 0. `pd.get_dummies` с
`drop_first=True` маха първата категория по азбучен ред.

In [ ]:
region = pd.get_dummies(credit["Region"], drop_first=True).astype(int)
display(region.head())

region_model = LinearRegression().fit(region, credit["Balance"])
print("β̂₀ =", round(region_model.intercept_, 2))
print(pd.Series(region_model.coef_, index=region.columns).round(2))
print("Среден баланс по региони:")
print(credit.groupby("Region")["Balance"].mean().round(2))

Свободният член е средният баланс в базовата категория `East`.
Коефициентите на индикаторите измерват разликите спрямо нея: в `South`
балансът е средно с 12.50 по-нисък, а в `West` с 18.69. В лекцията и двете
разлики не са значими.

In [ ]:
# --- ДА ОПИТАМЕ 5 ---
#
# Според лекцията изборът на базово ниво променя параметризацията, но не и
# прогнозите. Направете West базова категория: махнете колоната West вместо
# East от pd.get_dummies(credit["Region"]). Как се променят свободният член и
# коефициентите? Еднакви ли са прогнозите на двата модела (np.allclose)?

solution_2.to_test_5(credit, region_model, region)

<a id="chast-6"></a>
## Част 6. Взаимодействие

<a id="t18"></a>
### 18. Когато ефектът на TV зависи от radio

Моделът дотук събира отделните приноси на `TV` и `radio`: ефектът на TV е
един и същ при всеки бюджет за радио. Лекцията допуска, че при по-високи
радиоразходи ефектът от TV рекламата расте. Тогава комбинирането на медии
увеличава продажбите повече от сбора на отделните ефекти. Това се нарича
**взаимодействие** или **синергия**.

Създаваме нов признак `TV_radio = TV × radio`. Без `newspaper` моделът от
лекцията е

`sales = β₀ + β₁ · TV + β₂ · radio + β₃ · (TV × radio) + ε`

Ефектът на TV е `β₁ + β₃ · radio`, а ефектът на radio е `β₂ + β₃ · TV`.
Произведението се смята еднакво за тренировъчните и тестовите редове.

In [ ]:
def with_interaction(X):
    """TV, radio и произведението им TV_radio."""
    result = X[["TV", "radio"]].copy()
    result["TV_radio"] = X["TV"] * X["radio"]
    return result


additive_model = LinearRegression().fit(X_train[["TV", "radio"]], y_train)
interaction_model = LinearRegression().fit(with_interaction(X_train), y_train)
interaction_test_pred = interaction_model.predict(with_interaction(X_test))
interaction_coefficients = pd.Series(
    interaction_model.coef_, index=["TV", "radio", "TV_radio"]
)
print("Свободен член:", round(interaction_model.intercept_, 4))
print("Коефициенти с взаимодействие:")
print(interaction_coefficients.round(4))

comparison.loc["TV + radio"] = metric_row(
    "TV + radio", y_test, additive_model.predict(X_test[["TV", "radio"]])
)
comparison.loc["TV + radio + TV × radio"] = metric_row(
    "TV + radio + TV × radio", y_test, interaction_test_pred
)
display(comparison.round(3))

Взаимодействието може да подобри прогнозите за тези тестови пазари.
Това не доказва общо правило за всяка рекламна кампания. Сравнихме
предварително зададени модели върху едно разделяне; тестовият набор
не трябва да се използва за многократно нагласяване на модела.

### 19. Йерархичен принцип

Когато в модела има взаимодействие, лекцията препоръчва да се включат и
двата **главни ефекта**, `TV` и `radio`, дори когато p-стойността на някой
от тях не е значима. Без тях взаимодействието зависи от произволния избор
на нулата на скалата. Това е **йерархичният принцип**.

In [ ]:
# --- ДА ОПИТАМЕ 6 ---
#
# Обучете върху всички 200 пазара модел само с произведението TV × radio, без
# главните ефекти. После преместете нулата на radio: radio - 10 е бюджетът над
# 10 хиляди долара. Обучете отново с TV × (radio - 10). Как се променят R² и
# прогнозите? Повторете с йерархичния модел с TV, radio и TV × radio.

solution_2.to_test_6(df)

<a id="t20"></a>
### 20. Връзка, която линейният модел не вижда

Линейният модел вижда само колоните, които получава. Пример с четири точки
показва колко важно е това. Отговорът y е 1, когато точно един от двата
признака е 1, и 0 иначе. В логиката това се нарича "изключващо" или (XOR):

| x₁ | x₂ | y |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Отговорът се определя напълно от x₁ и x₂. Следващата клетка обучава два
модела: само с x₁ и x₂ и с добавено произведение x₁ · x₂, както `TV × radio`
в [точка 18 "Когато ефектът на TV зависи от radio"](#t18).

In [ ]:
xor = pd.DataFrame({"x1": [0, 0, 1, 1], "x2": [0, 1, 0, 1], "y": [0, 1, 1, 0]})
xor["x1_x2"] = xor["x1"] * xor["x2"]
print("Корелация с y: x1", xor["x1"].corr(xor["y"]), "| x2", xor["x2"].corr(xor["y"]))

for name, columns in [("без произведение", ["x1", "x2"]),
                      ("с произведение", ["x1", "x2", "x1_x2"])]:
    xor_model = LinearRegression().fit(xor[columns], xor["y"])
    # + 0.0 превръща -0.0, остатък от закръгляването, в 0.0
    xor["ŷ " + name] = xor_model.predict(xor[columns]).round(2) + 0.0
    print(name + ": β̂₀ =", round(xor_model.intercept_, 2) + 0.0,
          "| коефициенти", xor_model.coef_.round(2) + 0.0,
          "| R² =", round(xor_model.score(xor[columns], xor["y"]), 2))

display(xor)

Без произведението двата коефициента са 0 и моделът прогнозира 0.5 за
всяка точка, т.е. средното на y. R² е 0: моделът не е по-добър от
постоянната прогноза. Корелацията на всеки признак с y също е 0, макар че
y зависи изцяло от тях. Корелацията и правата виждат само линейна връзка.

С произведението моделът е y = x₁ + x₂ - 2 · x₁x₂ и познава и четирите
точки (R² = 1). При x₁ = x₂ = 1 двете единици се събират до 2, а членът
-2 · x₁x₂ връща прогнозата до 0. Произведението е взаимодействие: ефектът
на x₁ е 1 - 2 · x₂, т.е. +1 при x₂ = 0 и -1 при x₂ = 1. Ефектът сменя
посоката си, затова без произведението средно е 0.

Тук правилната колона се налучква лесно. При реални данни с много
признаци формата на зависимостта обикновено не е известна. Гъвкавите
методи от края на лекцията, като сплайни и дървета, научават формата от
данните. При класификация (Тема 4) същите четири точки не могат да се
разделят на два класа с права линия.

<a id="obobshtenie"></a>
## Обобщение

1. Простата линейна регресия е `Y ≈ β₀ + β₁X + ε`. Методът на най-малките
   квадрати избира β̂₀ и β̂₁, които минимизират RSS. `fit` научава
   коефициентите от тренировъчните редове, а `predict` дава прогнози за
   нови редове.
2. Остатъкът е действително минус прогнозирано. RSE е типичното
   отклонение от правата, а R² е делът от изменчивостта, обяснен от
   модела. При един предиктор R² = r².
3. Стандартната грешка показва колко се мени коефициентът при повторни
   извадки. Когато интервалът β̂ ± 2 · SE не съдържа нулата, данните
   показват връзка.
4. MAE, RMSE и R² върху отделени редове показват грешката за нови примери.
   Сравняваме ги и с постоянна прогноза. Графиката на остатъците показва
   какво моделът не описва.
5. При няколко предиктора коефициентът се тълкува при фиксирани останали.
   Корелирани предиктори могат да променят коефициента спрямо простата
   регресия, както radio и newspaper. Изборът напред добавя предикторите
   един по един.
6. Качествен предиктор с q нива става q - 1 индикаторни променливи.
   Коефициентите са разлики спрямо базовата категория, а изборът ѝ не
   променя прогнозите.
7. Взаимодействието позволява ефектът на един предиктор да зависи от друг.
   По йерархичния принцип главните ефекти остават в модела. Линейният
   модел вижда само колоните, които получава: без произведението той не
   вижда дори пълната зависимост в XOR.
8. Връзка в наблюдавани данни не е доказателство за причинен ефект.

По-нататък линейният модел се разширява за класификация: логистичната
регресия от Тема 4 прекарва линейния резултат през сигмоидна функция и
връща вероятност.

<a id="zadacha"></a>
## Задача

В лекцията ("Нелинейни ефекти" и "Полиномиална регресия") зависимостта
между разхода на гориво `mpg` (мили на галон) и мощността `horsepower` в
данните Auto е крива. Линейният модел може да я опише с **полиномиален
член**: нова колона `horsepower²`, точно както `TV × radio` в
[точка 18 "Когато ефектът на TV зависи от radio"](#t18).
Моделът остава линеен по коефициентите:

`mpg = β₀ + β₁ · horsepower + β₂ · horsepower² + ε`

Файлът `data/auto.csv` съдържа 397 модела коли. Колоната `name` е името на
модела, а `origin` е кодът на произхода: 1 за американски, 2 за
европейски и 3 за японски коли.

### 1. Зареждане и почистване

Заредете файла. В колоната `horsepower` липсващите стойности са записани
като `?`. Колко са те? Махнете тези редове и покажете `mpg` спрямо
`horsepower` на графика.

In [ ]:
# auto = pd.read_csv("data/auto.csv", na_values=...)
# print("Липсващи стойности:", ...)
# auto = auto.dropna(...)
# print("Коли:", len(auto))
# графика на mpg спрямо horsepower

### 2. Права

Обучете `LinearRegression` с `horsepower` върху всички коли, както в
лекцията. Отпечатайте коефициентите и R². Начертайте остатъците спрямо
прогнозите. Каква форма имат?

In [ ]:
# line_model = LinearRegression().fit(auto[["horsepower"]], auto["mpg"])
# print(line_model.intercept_, line_model.coef_, line_model.score(...))
# line_residuals = ...
# графика на остатъците спрямо прогнозите

### 3. Квадратичен член

Добавете колоната `horsepower²` и обучете отново. Как се променят R² и
графиката на остатъците? Сравнете с лекцията.

In [ ]:
# auto["horsepower2"] = auto["horsepower"] ** 2
# quad_model = LinearRegression().fit(auto[["horsepower", "horsepower2"]], auto["mpg"])
# ...

### 4. Проверка върху нови коли

Разделете колите на тренировъчни и тестови (80/20, `random_state=42`).
Обучете двата модела върху тренировъчните и сравнете средната абсолютна
грешка върху тестовите с постоянна прогноза (`DummyRegressor`).

In [ ]:
# auto_train, auto_test = train_test_split(auto, test_size=0.2, random_state=42)
# ...
# print("Тестова MAE:", ...)